# Andamento dei bond europei e americani

Dati da `data/rendimenti.csv` (aggiornalo con `python src/main.py`). Rendimenti in %, spread in punti percentuali.

- **EUR** = curva dei titoli di Stato AAA dell'area euro (BCE), giornaliera
- **DE/IT/FR/ES** = titoli di Stato per paese a 1 (non per ES), 2, 5, 10 e 30 anni (CNBC): giornalieri dal 2024, settimanali prima
- **US** = Treasury (FRED), giornalieri
- **spread corporate** = ICE BofA option-adjusted spread su FRED: solo gli ultimi 3 anni

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent / "src"))
from main import carica

DAL = "2015-01-01"  # inizio dei grafici
df = carica()
linee = lambda cols, titolo, dal=DAL: px.line(df.loc[dal:, cols].dropna(how="all").ffill(), title=titolo,
                                              labels={"value": "%", "Data": "", "variable": ""}).show()
df.index[-1].date()

datetime.date(2026, 11, 2)

## Rendimenti a 10 anni: USA, area euro e paesi

In [2]:
linee(["US 10Y", "EUR 10Y", "DE 10Y", "IT 10Y", "FR 10Y", "ES 10Y"], "Rendimento a 10 anni")

## Breve e lungo termine: 2 anni contro 30 anni

In [3]:
for paese in ("US", "EUR", "IT", "DE", "FR", "ES"):
    linee([f"{paese} 2Y", f"{paese} 30Y"], f"{paese}: 2 anni e 30 anni", dal="2022-01-01")

## Curva dei rendimenti: oggi, 1 anno fa, 3 anni fa
Asse in anni alla scadenza (scala logaritmica).

In [4]:
anni = lambda k: int(k[:-1]) / (12 if k[-1] == "M" else 1)  # '3M' -> 0.25, '10Y' -> 10
curve = []
for area in ("US", "EUR", "IT", "DE", "FR", "ES"):
    cols = [c for c in df if c.startswith(area + " ") and c.split()[1][:-1].isdigit()]  # 'IT 10Y' sì, 'US IG spread' no
    s = df[cols].dropna()
    for quando in (0, 1, 3):
        r = s.asof(s.index[-1] - pd.DateOffset(years=quando))
        curve += [{"Area": area, "Quando": f"{quando} anni fa" if quando else "oggi", "Anni": anni(c.split()[1]), "%": r[c]}
                  for c in cols]
px.line(pd.DataFrame(curve), x="Anni", y="%", color="Quando", facet_col="Area", facet_col_wrap=3, markers=True,
        log_x=True, height=650, title="Curva dei rendimenti").show()

## Inclinazione della curva (10 anni - 2 anni)
Sotto zero = curva invertita: storicamente anticipa un rallentamento.

In [5]:
fig = px.line(df.loc[DAL:, ["US 10Y-2Y", "EUR 10Y-2Y"]].dropna(how="all").ffill(), title="10Y - 2Y",
              labels={"value": "punti %", "Data": "", "variable": ""})
fig.add_hline(y=0, line_dash="dot")
fig.show()

## Spread verso il Bund e USA - area euro

In [6]:
linee(["IT-DE 10Y", "FR-DE 10Y", "ES-DE 10Y", "US-EUR 10Y"], "Spread a 10 anni")

## Spread corporate (sopra i titoli di Stato)

In [7]:
linee(["US IG spread", "US HY spread", "EUR HY spread"], "Spread corporate (option-adjusted)", dal=None)

## Riepilogo
Variazioni in punti base; percentile = posizione dell'ultimo valore negli ultimi 10 anni (1 = massimo).

In [8]:
pd.read_excel(Path.cwd().parent / "data" / "riepilogo.xlsx")

,Serie,Dati al,Ultimo %,Var 1m (pb),Var 3m (pb),Var 1a (pb),Min 10a,Max 10a,Percentile 10a
0,US 3M,2026-10-30,4.160,-4.000,34.000,24.000,0.000,5.630,0.705
1,US 2Y,2026-10-30,4.560,-32.000,33.000,95.000,0.090,5.190,0.890
2,US 5Y,2026-10-30,4.710,-38.000,33.000,99.000,0.190,5.090,0.979
3,US 10Y,2026-10-30,4.940,-35.000,26.000,83.000,0.520,5.310,0.986
4,US 30Y,2026-10-30,5.310,-33.000,10.000,66.000,0.990,5.670,0.988
5,US IG spread,2026-10-30,0.940,10.000,14.000,17.000,0.730,1.290,0.771
6,US HY spread,2026-10-30,3.440,32.000,60.000,59.000,2.590,4.610,0.879
7,EUR HY spread,2026-10-30,3.590,59.000,94.000,80.000,2.480,4.650,0.848
8,EUR 3M,2026-10-30,2.556,5.513,15.200,64.292,-0.930,3.913,0.803
9,EUR 2Y,2026-10-30,3.005,-15.406,29.203,103.856,-0.971,3.257,0.949
